In [1]:
pip install undetected-chromedriver

  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Created wheel for undetected-chromedriver: filename=undetected_chromedriver-3.5.5-py3-none-any.whl size=47215 sha256=4b24679e099bba1de5763e0ab794474c297c230fd4a85a6bc96658bf40241371
  Stored in directory: c:\users\asus m1603\appdata\local\pip\cache\wheels\7a\5f\c1\06f68421cc7172ef51504631252870bcb3a2fdf3b6a025f362
Successfully built undetected-chromedriver

   ---------------------------------------- 0/2 [websockets]
   ---------------------------------------- 0/2 [websockets]
   ---------------------------------------- 0/2 [websockets]
   ---------------------------------------- 0/2 [websockets]
   ---------------------------------------- 0/2 [websockets]
   

In [2]:
pip install selenium-stealth

Note: you may need to restart the kernel to use updated packages.


In [3]:
pip install python-anticaptcha

Note: you may need to restart the kernel to use updated packages.


In [1]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait as W
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.keys import Keys
import time

try:
    import undetected_chromedriver as uc
    USE_UC = True
except Exception:
    USE_UC = False

options = webdriver.ChromeOptions()
options.add_argument("--start-maximized")

if USE_UC:
    uc_opts = uc.ChromeOptions()
    for a in options.arguments:
        uc_opts.add_argument(a)
    for k, v in getattr(options, "_experimental_options", {}).items():
        uc_opts.add_experimental_option(k, v)
    uc_opts.add_argument("--lang=en-US,en")
    driver = uc.Chrome(options=uc_opts)
    try:
        driver.execute_cdp_cmd("Page.addScriptToEvaluateOnNewDocument", {
            "source": """
                Object.defineProperty(navigator, 'webdriver', {get: () => undefined});
                Object.defineProperty(navigator, 'languages', {get: () => ['en-US','en']});
                Object.defineProperty(navigator, 'plugins', {get: () => [1,2,3]});
                window.chrome = window.chrome || { runtime: {} };
            """
        })
    except Exception:
        pass
else:
    driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

def is_cloudflare_present(driver):
    try:
        title = driver.title or ""
        if "Attention Required" in title or "Just a moment" in title or "Checking" in title:
            return True
    except Exception:
        pass
    try:
        if driver.find_elements(By.CSS_SELECTOR, "div[id*='cf-content'], div[id*='cf-wrapper'], iframe[src*='captcha'], iframe[src*='recaptcha']"):
            return True
    except Exception:
        pass
    return False

def wait_manual_captcha(driver, timeout=120):
    if not is_cloudflare_present(driver):
        return True
    print(f"[!] Cloudflare/recaptcha tespit edildi. Manuel çözmeni bekliyorum (timeout {timeout}s)")
    start = time.time()
    while time.time() - start < timeout:
        if not is_cloudflare_present(driver):
            print("[+] Cloudflare temizlendi.")
            return True
        time.sleep(2)
    print("[-] Manuel çözüm süresi doldu.")
    return False

def accept_cookies_if_present(driver, timeout=3):
    try:
        btn = driver.find_element(By.CSS_SELECTOR, "#onetrust-accept-btn-handler")
        try:
            btn.click()
            return True
        except Exception:
            try:
                driver.execute_script("arguments[0].click();", btn)
                return True
            except Exception:
                return False
    except Exception:
        return False

try:
    driver.get("https://www.mastersportal.com/")
    W(driver, 20).until(lambda d: d.title != "")

    if is_cloudflare_present(driver):
        wait_manual_captcha(driver, timeout=120)

    EMAIL = "oejjwfrkromahdicsr@fxavaj.com"
    PASSWORD = "wKubN8AGCNV-L8r"

    def lower_iframes_zindex(driver):
        try:
            driver.execute_script("""
              for (const f of document.querySelectorAll('iframe')) {
                f.style.pointerEvents = 'none';
                f.style.zIndex = '-1';
              }
            """)
        except Exception:
            pass

    def get_active_modal_root(driver, timeout=6):
        sel = "div[role='dialog'], .ReactModal__Content, .modal, .Modal, .sp-modal, .SpModal"
        try:
            return W(driver, timeout).until(EC.presence_of_element_located((By.CSS_SELECTOR, sel)))
        except Exception:
            return None

    def safe_click(driver, el):
        driver.execute_script("arguments[0].scrollIntoView({block:'center'});", el)
        time.sleep(0.05)
        driver.execute_script("arguments[0].click();", el)

    def login_mastersportal(driver, email, password, timeout=15):
        driver.get("https://www.mastersportal.com/account/?section=recommendations")
        W(driver, timeout).until(lambda d: d.execute_script("return document.readyState") == "complete")
        lower_iframes_zindex(driver)

        if is_cloudflare_present(driver):
            if not wait_manual_captcha(driver, timeout=120):
                return False

        try:
            accept_cookies_if_present(driver, timeout=3)
        except Exception:
            pass

        openers = [
            "button.GoToProfilePage",
            "button.DriverButton.GoToProfilePage",
            "a[href*='account'], button.login, a.login"
        ]
        for css in openers:
            try:
                el = W(driver, 3).until(EC.element_to_be_clickable((By.CSS_SELECTOR, css)))
                safe_click(driver, el)
                break
            except Exception:
                continue

        root = get_active_modal_root(driver, timeout=5)
        if root:
            try:
                btn_login_in_modal = W(driver, 4).until(
                    EC.element_to_be_clickable((By.CSS_SELECTOR, ".GoToLoginWrapper button"))
                )
                safe_click(driver, btn_login_in_modal)
                time.sleep(0.2)
            except Exception:
                pass

        def find_form_scope():
            r = get_active_modal_root(driver, timeout=3)
            scope = r if r else driver.find_element(By.TAG_NAME, "body")
            try:
                form = scope.find_element(By.CSS_SELECTOR, "form.FgForm")
                return form
            except Exception:
                return None

        form = None
        for _ in range(10):
            form = find_form_scope()
            if form:
                break
            time.sleep(0.3)
        if not form:
            return False

        def q(scope, css, wait=6):
            return W(driver, wait).until(EC.visibility_of_element_located((By.CSS_SELECTOR, css)))

        try:
            email_el = q(form, 'input[data-cy="EmailInput"], input[name="Email"]', wait=10)
            pass_el  = q(form, 'input[data-cy="PasswordInput"], input[name="Password"]', wait=10)
        except Exception:
            return False

        try:
            email_el.clear()
        except Exception:
            pass
        email_el.send_keys(email)

        try:
            pass_el.clear()
        except Exception:
            pass
        pass_el.send_keys(password)

        try:
            submit_btn = form.find_element(By.CSS_SELECTOR, 'button[data-cy="SubmitButton"], button[type="submit"]')
            safe_click(driver, submit_btn)
        except Exception:
            try:
                pass_el.send_keys(Keys.ENTER)
            except Exception:
                pass
            try:
                driver.execute_script(
                    "arguments[0].requestSubmit ? arguments[0].requestSubmit() : arguments[0].submit();", form
                )
            except Exception:
                pass

        try:
            W(driver, 12).until(EC.url_contains("/account/"))
            return True
        except Exception:
            try:
                W(driver, 6).until(
                    EC.presence_of_element_located(
                        (By.CSS_SELECTOR, "a[href*='/account/'], .profile-menu, .user-avatar")
                    )
                )
                return True
            except Exception:
                return False

    ok = login_mastersportal(driver, EMAIL, PASSWORD)
    print("login:", ok)

finally:
    pass


login: True


In [ ]:
URL = "https://www.mastersportal.com/studies/66677/computer-vision-robotics-and-machine-learning.html?ref=search_card"

import json, time, re
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait as _Wait
from selenium.webdriver.support import expected_conditions as EC

AFTER_PAGE_LOAD_SEC = 10
wait = _Wait(driver, 20)

# -------------------- Yardımcılar --------------------
def _norm_text(s):
    if s is None: return None
    s = str(s).replace("\xa0"," ")
    s = re.sub(r"\s+"," ",s).strip()
    return s or None

def safe_get_text(obj, by=None, sel=None):
    try:
        el = wait.until(EC.presence_of_element_located((by, sel))) if sel else obj
        return _norm_text(el.text)
    except Exception:
        return None

def safe_find_all(by, sel, ctx=None):
    try:
        return (ctx or driver).find_elements(by, sel)
    except Exception:
        return []

def safe_attr(we, name):
    try:
        v = we.get_attribute(name); return _norm_text(v)
    except Exception:
        return None

def deep_scroll(rounds=8, pause=0.6):
    """Lazy-load bloklarını tetiklemek için aşağı/yukarı gezinti."""
    last = 0
    for _ in range(rounds):
        driver.execute_script("window.scrollBy(0, Math.max(500, Math.floor(window.innerHeight*0.95)));")
        time.sleep(pause)
        h = driver.execute_script("return document.body.scrollHeight || 0;")
        if h == last:
            driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
            time.sleep(0.4)
            driver.execute_script("window.scrollTo(0, 0);")
            time.sleep(0.4)
            h2 = driver.execute_script("return document.body.scrollHeight || 0;")
            if h2 == h:
                break
            last = h2
        else:
            last = h

def get_degree_tag():
    for sel in ['div.DegreeTags.js-degreeTags span.Tag.js-tag','div.DegreeTags span.Tag']:
        try:
            el = driver.find_element(By.CSS_SELECTOR, sel); t = safe_get_text(el)
            if t: return t
        except Exception: pass
    return None

def get_university_and_program():
    uni=prog=None
    try:
        h1 = driver.find_element(By.CSS_SELECTOR, 'h1.StudyTitleWrapper, .StudyTitleWrapper')
        try: prog = safe_get_text(h1.find_element(By.CSS_SELECTOR, '.StudyTitle a, .StudyTitle'))
        except Exception: pass
        try: uni = safe_get_text(h1.find_element(By.CSS_SELECTOR, '.OrganisationName a, .OrganisationName'))
        except Exception: pass
    except Exception: pass
    return uni, prog

def get_review():
    score=count=None
    try:
        wrap = driver.find_element(By.CSS_SELECTOR, '#js-reviewsReadMoreButton, .ReviewReadMoreButton')
        score = safe_get_text(wrap.find_element(By.CSS_SELECTOR, '.Value'))
        cnt_txt=None
        for sel in ['.ReadReviews','.ReviewQuantity']:
            els = wrap.find_elements(By.CSS_SELECTOR, sel)
            if els: cnt_txt = safe_get_text(els[0]); break
        if cnt_txt:
            m = re.search(r'(\d[\d,\.]*)', cnt_txt); count = m.group(1).replace(",","") if m else None
    except Exception:
        try:
            score = safe_get_text(driver.find_element(By.CSS_SELECTOR, '.AverageStarRating'))
            count = safe_get_text(driver.find_element(By.CSS_SELECTOR, '.ReviewQuantity'))
            if count: count = re.sub(r'[^\d]','',count)
        except Exception: pass
    return score, count

def get_durations():
    """
    <article class="FactItem"><h3 class="FactItemTitle">Duration</h3> ... <ul class="FactList DurationList">...</ul>
    """
    out = []
    try:
        art = driver.find_element(
            By.XPATH,
            "//article[contains(@class,'FactItem')][.//h3[contains(@class,'FactItemTitle') and normalize-space()='Duration']]"
        )
        items = art.find_elements(By.CSS_SELECTOR, "ul.FactList.DurationList > li.FactListItem")
        for li in items:
            mode = None
            try:
                mode = safe_get_text(li.find_element(By.CSS_SELECTOR, ".FactItemInformation.FactListTitle"))
            except Exception:
                pass
            vals = []
            for d in li.find_elements(By.CSS_SELECTOR, ".FactListSubList .Duration"):
                dd = safe_attr(d, "data-duration")
                dp = safe_attr(d, "data-period")
                txt = safe_get_text(d)
                if dd and dp: vals.append(f"{dd} {dp}")
                elif txt:     vals.append(txt)
            if mode and vals:
                out.append(f"{mode} {' / '.join(vals)}")
    except Exception:
        pass
    return out

def get_start_dates():
    """
    #js-StartdateContainer .StartDateItem -> Starting <time datetime> + DeadlinesList <time datetime>
    """
    res = []
    try:
        cont = driver.find_element(By.CSS_SELECTOR, "#js-StartdateContainer")
        driver.execute_script("arguments[0].scrollIntoView({block:'center'});", cont)
        time.sleep(0.2)
        for li in cont.find_elements(By.CSS_SELECTOR, "li.StartDateItem"):
            start_txt = start_dt = deadline_txt = deadline_dt = None
            try:
                t = li.find_element(By.CSS_SELECTOR, ".StartDateItemTime time")
                start_txt = safe_get_text(t)
                start_dt  = safe_attr(t, "datetime")
            except Exception:
                pass
            try:
                t2 = li.find_element(By.CSS_SELECTOR, ".DeadlinesList .ApplicationDeadline time, .DeadlinesList .Deadline time")
                deadline_txt = safe_get_text(t2)
                deadline_dt  = safe_attr(t2, "datetime")
            except Exception:
                pass
            if start_txt:
                label = f"Starting {start_txt}"
                if deadline_txt:
                    label += f" | Apply before {deadline_txt}"
                meta = []
                if start_dt:    meta.append(f"start_dt={start_dt}")
                if deadline_dt: meta.append(f"deadline_dt={deadline_dt}")
                if meta: label += f" ({', '.join(meta)})"
                res.append(label)
    except Exception:
        pass
    return res

def get_language():
    """
    <article class="FactItem LanguageFact"> ... <div class="Languages FactItemInformation">English</div>
    """
    try:
        art = driver.find_element(
            By.XPATH,
            "//article[contains(@class,'FactItem') and contains(@class,'LanguageFact')][.//h3[contains(@class,'FactItemTitle') and normalize-space()='Language']]"
        )
        val = safe_get_text(art.find_element(By.CSS_SELECTOR, ".Languages.FactItemInformation"))
        if val: return val
    except Exception:
        pass
    return None

def get_language_requirements():
    """
    section.EnglishRequirementsCards .CardContents:
      Heading (+ optional SubHeading) + .Score span -> "IELTS 6.5", "TOEFL IBT 88", "PTE Academic 67", "Duolingo English Test 120"
    """
    out = []
    scopes = driver.find_elements(By.CSS_SELECTOR, "section.EnglishRequirementsCards .CardContents") \
          or driver.find_elements(By.CSS_SELECTOR, ".EnglishRequirementsCards .CardContents") \
          or driver.find_elements(By.CSS_SELECTOR, ".js-englishRequirementsContainerForKeyFacts .CardContents")
    for card in scopes:
        try:
            heading = ""
            try:
                heading = safe_get_text(card.find_element(By.CSS_SELECTOR, ".Heading")) or ""
            except Exception:
                pass
            sub = ""
            try:
                sub = safe_get_text(card.find_element(By.CSS_SELECTOR, ".SubHeading")) or ""
            except Exception:
                pass
            score = ""
            try:
                score = safe_get_text(card.find_element(By.CSS_SELECTOR, ".Score span")) or ""
            except Exception:
                pass
            label = heading.strip()
            if "TOEFL" in (label.upper()):
                label = f"TOEFL {sub.strip()}" if sub else "TOEFL"
            if label and score:
                out.append(f"{label} {score}".strip())
        except Exception:
            continue
    return out

def get_delivery_type():
    """
    <article class="FactItem"><h3>Delivered</h3><div class="FactItemInformation">On Campus</div>
    """
    try:
        art = driver.find_element(
            By.XPATH,
            "//article[contains(@class,'FactItem')][.//h3[contains(@class,'FactItemTitle') and normalize-space()='Delivered']]"
        )
        return safe_get_text(art.find_element(By.CSS_SELECTOR, ".FactItemInformation"))
    except Exception:
        return None

def get_credits():
    """
    <article class="FactItem"><h3>Credits</h3><div class="FactItemInformation">180 alternative credits</div>
    """
    try:
        art = driver.find_element(
            By.XPATH,
            "//article[contains(@class,'FactItem')][.//h3[contains(@class,'FactItemTitle') and normalize-space()='Credits']]"
        )
        return safe_get_text(art.find_element(By.CSS_SELECTOR, ".FactItemInformation"))
    except Exception:
        return None

def get_campus_location():
    """
    <article class="FactItem"><h3>Campus Location</h3><ul class="CampusLocationList"><li>...</li></ul>
    """
    try:
        art = driver.find_element(
            By.XPATH,
            "//article[contains(@class,'FactItem')][.//h3[contains(@class,'FactItemTitle') and normalize-space()='Campus Location']]"
        )
        vals = [safe_get_text(li) for li in art.find_elements(By.CSS_SELECTOR, ".CampusLocationList li")]
        vals = [v for v in vals if v]
        return vals or []
    except Exception:
        return []

def get_disciplines():
    """
    <article class="FactItem Disciplines"> ... <a class="TextOnly">Computer Sciences</a> ...
    """
    out = []
    try:
        art = driver.find_element(By.CSS_SELECTOR, "article.FactItem.Disciplines, article.Disciplines")
        for a in art.find_elements(By.CSS_SELECTOR, "a.TextOnly, a"):
            t = safe_get_text(a)
            if t:
                if t.lower().startswith("view ") and " other " in t.lower():
                    continue
                out.append(t)
    except Exception:
        pass
    return out

def get_gpa_class():
    """
    (Varsa) #OtherRequirements / EntryRequirements altında 'Lower/Upper/First/Second/GPA/Class' içeren .Score span
    """
    try:
        scopes = driver.find_elements(By.CSS_SELECTOR, '#OtherRequirements, article#OtherRequirements, section#EntryRequirements, .Requirements, .RequirementsWrapper') or [driver]
        for sc in scopes:
            for sp in safe_find_all(By.CSS_SELECTOR, '.Score span, .Score', ctx=sc):
                t=safe_get_text(sp)
                if t and re.search(r'Lower|Upper|First|Second|GPA|Class', t, re.I): return t
    except Exception: pass
    return None

def get_requirements_text():
    """
    Bu sayfada Start dates altında 'More details' metni var; ayrıca eski fallback'ler korunur.
    """
    out = []
    try:
        more = driver.find_elements(By.CSS_SELECTOR, ".FactItemInformation.MoreDeadlineDetailsText p")
        for p in more:
            t = safe_get_text(p)
            if t and t not in out:
                out.append(t)
    except Exception:
        pass
    for sel in [
        'article#OtherRequirements .OtherRequirementsContent',
        'section#EntryRequirements',
        '.RequirementsConditions',
        '.Requirements, .RequirementsWrapper'
    ]:
        for box in safe_find_all(By.CSS_SELECTOR, sel):
            txt = safe_get_text(box)
            if txt and txt not in out:
                out.append(txt)
    for li in safe_find_all(By.CSS_SELECTOR, 'article#OtherRequirements li, section#EntryRequirements li'):
        t = safe_get_text(li)
        if t and t not in out:
            out.append(t)
    return out

def get_tuition():
    res={"original_amount":None,"original_currency":None,"display_amount":None,"display_currency":None,"display_duration":None}
    try:
        amount = driver.find_element(By.CSS_SELECTOR, '.Amount .js-currencyAmount, .TuitionFee .js-currencyAmount, .TuitionValue .js-currencyAmount')
        res["original_amount"]=safe_attr(amount,'data-original-amount')
        res["original_currency"]=safe_attr(amount,'data-currency')
        res["display_amount"]=safe_get_text(amount)
        cur=None
        for sel in ['.Amount .CurrencyType[data-currency-text]','.Amount .CurrencyType']:
            els=driver.find_elements(By.CSS_SELECTOR, sel)
            if els: cur=safe_get_text(els[0]); break
        res["display_currency"]=cur if cur else None
        els=driver.find_elements(By.CSS_SELECTOR, '.Amount .CurrencyType ~ .CurrencyType')
        if els: res["display_duration"]=safe_get_text(els[0])
    except Exception:
        res["display_amount"]=safe_get_text(driver, By.CSS_SELECTOR, '.TuitionValue')
    return res

def get_cost_of_living():
    out={"amount_min_display":None,"amount_max_display":None,"base_currency":None,"amount_min_base":None,"amount_max_base":None,"display_currency":None}
    try:
        costs=driver.find_element(By.CSS_SELECTOR, '.Costs')
        spans=costs.find_elements(By.CSS_SELECTOR, '.Amount span[data-amount]')
        if spans:
            out["base_currency"]=safe_attr(spans[0],'data-currency')
            out["amount_min_base"]=safe_attr(spans[0],'data-amount')
            out["amount_min_display"]=safe_get_text(spans[0])
            if len(spans)>1:
                out["amount_max_base"]=safe_attr(spans[1],'data-amount')
                out["amount_max_display"]=safe_get_text(spans[1])
        cur=None
        els=costs.find_elements(By.CSS_SELECTOR, '.CurrencyType, [data-currency-text]')
        if els: cur=safe_get_text(els[0])
        out["display_currency"]=cur
    except Exception: pass
    return out

def record_from_url(url):
    rec={"source_url":url,"degree_tag":None,"university_name":None,"program_name":None,"review_score":None,"review_count":None,"durations":[], \
         "start_dates":[], "language":None,"language_requirements":[], "delivery_type":None,"credits":None,"campus_location":[], \
         "disciplines":[], "gpa_class":None,"requirements_text":[], "tuition":None,"cost_of_living":None,"error":None}
    try:
        driver.get(url)
        time.sleep(AFTER_PAGE_LOAD_SEC)
        deep_scroll()  
        try:
            wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, 'h1.StudyTitleWrapper, .StudyTitle')))
        except Exception: pass

        rec["degree_tag"]=get_degree_tag()
        uni,prog=get_university_and_program()
        rec["university_name"],rec["program_name"]=uni,prog
        rv,rc=get_review()
        rec["review_score"],rec["review_count"]=rv,rc
        rec["durations"]=get_durations()
        rec["start_dates"]=get_start_dates()
        rec["language"]=get_language()
        rec["language_requirements"]=get_language_requirements()
        rec["delivery_type"]=get_delivery_type()
        rec["credits"]=get_credits()
        rec["campus_location"]=get_campus_location() or []
        rec["disciplines"]=get_disciplines()
        rec["gpa_class"]=get_gpa_class()
        rec["requirements_text"]=get_requirements_text()
        rec["tuition"]=get_tuition()
        rec["cost_of_living"]=get_cost_of_living()
    except Exception as e:
        rec["error"]=f"{type(e).__name__}: {_norm_text(str(e))}"
    return rec

result = record_from_url(URL)
print(json.dumps(result, ensure_ascii=False, indent=2))


{
  "source_url": "https://www.mastersportal.com/studies/66677/computer-vision-robotics-and-machine-learning.html?ref=search_card",
  "degree_tag": "M.Sc.",
  "university_name": "University of Surrey",
  "program_name": "Computer Vision, Robotics and Machine Learning",
  "review_score": "4.4",
  "review_count": "72",
  "durations": [
    "Full-time 12 months",
    "Part-time 60 months"
  ],
  "start_dates": [
    "Starting February 2026 | Apply before Dec 2025 (start_dt=2026-02-02 00:00:00, deadline_dt=2025-12-10 00:00:00)",
    "Starting September 2026 | Apply before Jul 2026 (start_dt=2026-09-22 00:00:00, deadline_dt=2026-07-15 00:00:00)"
  ],
  "language": "English",
  "language_requirements": [
    "Duolingo English Test 120",
    "TOEFL IBT 88",
    "PTE Academic 67",
    "IELTS 6.5"
  ],
  "delivery_type": "On Campus",
  "credits": "180 alternative credits",
  "campus_location": [
    "Guildford, United Kingdom"
  ],
  "disciplines": [
    "Computer Sciences",
    "Artificial Int

In [2]:
import json, time, re, os
from pathlib import Path
import pandas as pd
from urllib.parse import urljoin
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait as _Wait
from selenium.webdriver.support import expected_conditions as EC

URL_START = URL_START = "https://www.mastersportal.com/search/master/data-science-big-data/europe?page=18"
EXCEL_PATH = r"C:\Users\asus M1603\Desktop\Final Project\Python\mastersportal_ai_all.xlsx"
CSV_PATH   = r"C:\Users\asus M1603\Desktop\Final Project\Python\mastersportal_ai_all.csv"

wait = _Wait(driver, 20)

def _stealth_patch():
    try:
        driver.execute_script("""
            try {
              Object.defineProperty(navigator, 'webdriver', {get: () => undefined});
            } catch(e){}
            try {
              if (!navigator.languages || navigator.languages.length === 0) {
                Object.defineProperty(navigator, 'languages', {get: () => ['en-US','en']});
              }
            } catch(e){}
            try {
              if (!navigator.plugins || navigator.plugins.length === 0) {
                const f = function(){};
                const fake = {length: 3, 0:f,1:f,2:f};
                Object.setPrototypeOf(fake, PluginArray.prototype);
                Object.defineProperty(navigator, 'plugins', {get: () => fake});
              }
            } catch(e){}
        """)
    except Exception:
        pass

def _looks_like_cf():
    try:
        title = (driver.title or "").lower()
    except Exception:
        title = ""
    if any(k in title for k in ["just a moment", "attention required", "checking your browser"]):
        return True
    sel_candidates = [
        '#cf-wrapper', '#challenge-running', 'div[data-translate="checking_browser"]',
        'iframe[src*="challenge"]', 'div[class*="cf-browser-verification"]',
        'div[id*="cf-challenge"]', 'div[class*="cf-challenge"]'
    ]
    try:
        for s in sel_candidates:
            if driver.find_elements(By.CSS_SELECTOR, s):
                return True
    except Exception:
        pass
    return False

def _wait_cf_clear(timeout=35):
    end = time.time() + timeout
    while time.time() < end:
        if not _looks_like_cf():
            try:
                rs = driver.execute_script("return document.readyState") or ""
            except Exception:
                rs = ""
            if rs.lower() == "complete":
                return True
        time.sleep(1.2)
    return False

def go(url, tries=6, per_try_wait=35):
    last_err = None
    for attempt in range(1, tries+1):
        try:
            driver.get(url)
            _stealth_patch()
            try:
                driver.execute_script("window.scrollBy(0, Math.max(2, Math.floor(window.innerHeight*0.1)));")
            except Exception:
                pass
            if _wait_cf_clear(timeout=per_try_wait):
                return True
            time.sleep(1.5 * attempt)
        except Exception as e:
            last_err = e
            time.sleep(1.5 * attempt)

    try:
        driver.refresh()
        _stealth_patch()
        if _wait_cf_clear(timeout=per_try_wait):
            return True
    except Exception as e:
        last_err = e
    if last_err:
        raise last_err
    raise RuntimeError("Cloudflare engeli aşılamadı")

def _norm_text(s):
    if s is None: return None
    s = str(s).replace("\xa0"," ")
    s = re.sub(r"\s+"," ",s).strip()
    return s or None

def safe_get_text(obj, by=None, sel=None):
    try:
        el = wait.until(EC.presence_of_element_located((by, sel))) if sel else obj
        return _norm_text(el.text)
    except Exception:
        return None

def safe_find_all(by, sel, ctx=None):
    try:
        return (ctx or driver).find_elements(by, sel)
    except Exception:
        return []

def safe_attr(we, name):
    try:
        v = we.get_attribute(name); return _norm_text(v)
    except Exception:
        return None

def scroll_lazy_until_cards_stop(min_pause=0.6, max_idle=4):
    last = 0; idle = 0
    while True:
        cards = driver.find_elements(By.CSS_SELECTOR, 'ul.SearchResultsList li.SearchResultItem a.SearchStudyCard.js-bestFitStudycard.js-studyCardExperiment.HoverEffect[href*="/studies/"]')
        n = len(cards)
        if n>last: idle=0; last=n
        else: idle+=1
        driver.execute_script("window.scrollBy(0, Math.max(500, Math.floor(window.innerHeight*0.95)));")
        time.sleep(min_pause)
        if idle>=max_idle: break
    return last

def is_ad_placeholder(li_we):
    try:
        li_we.find_element(By.CSS_SELECTOR, 'article[id^="div-gpt-ad-"]'); return True
    except Exception:
        return False

def get_degree_tag():
    for sel in ['div.DegreeTags.js-degreeTags span.Tag.js-tag','div.DegreeTags span.Tag']:
        try:
            el = driver.find_element(By.CSS_SELECTOR, sel); t = safe_get_text(el)
            if t: return t
        except Exception: pass
    return None

def get_university_and_program():
    uni=prog=None
    try:
        h1 = driver.find_element(By.CSS_SELECTOR, 'h1.StudyTitleWrapper, .StudyTitleWrapper')
        try: prog = safe_get_text(h1.find_element(By.CSS_SELECTOR, '.StudyTitle a, .StudyTitle'))
        except Exception: pass
        try: uni = safe_get_text(h1.find_element(By.CSS_SELECTOR, '.OrganisationName a, .OrganisationName'))
        except Exception: pass
    except Exception: pass
    return uni, prog

def get_review():
    score=count=None
    try:
        wrap = driver.find_element(By.CSS_SELECTOR, '#js-reviewsReadMoreButton, .ReviewReadMoreButton')
        score = safe_get_text(wrap.find_element(By.CSS_SELECTOR, '.Value'))
        cnt_txt=None
        for sel in ['.ReadReviews','.ReviewQuantity']:
            els = wrap.find_elements(By.CSS_SELECTOR, sel)
            if els: cnt_txt = safe_get_text(els[0]); break
        if cnt_txt:
            m = re.search(r'(\d[\d,\.]*)', cnt_txt); count = m.group(1).replace(",","") if m else None
    except Exception:
        try:
            score = safe_get_text(driver.find_element(By.CSS_SELECTOR, '.AverageStarRating'))
            count = safe_get_text(driver.find_element(By.CSS_SELECTOR, '.ReviewQuantity'))
            if count: count = re.sub(r'[^\d]','',count)
        except Exception: pass
    return score, count

def get_durations():
    out=[]
    try:
        for li in driver.find_elements(By.CSS_SELECTOR, 'ul.FactList.DurationList > li.FactListItem'):
            mode=None
            for sel in ['.FactListTitle','.js-durationFact']:
                els=li.find_elements(By.CSS_SELECTOR, sel)
                if els: mode = safe_get_text(els[0]); break
            subs = safe_find_all(By.CSS_SELECTOR, '.FactListSubList .Duration', ctx=li)
            vals = [safe_get_text(x) for x in subs if safe_get_text(x)]
            tip=None
            try:
                tip_icon = li.find_element(By.CSS_SELECTOR, '.DurationTooltip .InfoIcon')
                tip = safe_attr(tip_icon,'data-tooltip-text')
                if tip: tip = _norm_text(re.sub(r'<.*?>',' ', tip))
            except Exception: pass
            if mode and (vals or tip):
                text = f"{mode} {' / '.join(vals)}"
                if tip: text += f" | tooltip: {tip}"
                out.append(text)
    except Exception: pass
    if not out:
        v = safe_get_text(driver, By.CSS_SELECTOR, '.DurationValue')
        if v: out=[v]
    return out

def get_start_dates():
    res=[]
    try:
        for li in driver.find_elements(By.CSS_SELECTOR, '#js-StartdateContainer li.StartDateItem'):
            start=None
            for sel in ['.StartDateItemTime time','.StartDateItemTime']:
                els=li.find_elements(By.CSS_SELECTOR, sel)
                if els: start=safe_get_text(els[0]); break
            deadline=None
            for sel in ['.DeadlinesList .ApplicationDeadline time','.DeadlinesList .Deadline time','.DeadlinesList .Deadline']:
                els=li.find_elements(By.CSS_SELECTOR, sel)
                if els: deadline=safe_get_text(els[0]); break
            if start:
                label=f"Starting {start}"
                if deadline: label+=f" | Apply before {deadline}"
                res.append(label)
    except Exception: pass
    return res

def get_language():
    for sel in ['.Languages.FactItemInformation','article.FactItem .FactItemTitle+div.FactItemInformation']:
        try:
            txt=safe_get_text(driver, By.CSS_SELECTOR, sel)
            if txt: return txt
        except Exception: pass
    return None

def get_language_requirements():
    out=[]
    for card_sel in ['.EnglishRequirementsCards .CardContents','.js-englishRequirementsContainerForKeyFacts .CardContents']:
        for card in safe_find_all(By.CSS_SELECTOR, card_sel):
            try:
                head=[]
                for p in ['.Heading','.SubHeading']:
                    els=card.find_elements(By.CSS_SELECTOR, p)
                    if els:
                        t=safe_get_text(els[0])
                        if t: head.append(t)
                score=None
                els=card.find_elements(By.CSS_SELECTOR, '.Score span')
                if els: score=safe_get_text(els[0])
                if head and score: out.append(f"{' '.join(head)} {score}".strip())
            except Exception: continue
    return out

def get_delivery_type():
    try:
        for art in driver.find_elements(By.CSS_SELECTOR, 'article.FactItem'):
            ttl=safe_get_text(art.find_element(By.CSS_SELECTOR, '.FactItemTitle'))
            if ttl and ttl.lower().startswith('delivered'):
                return safe_get_text(art.find_element(By.CSS_SELECTOR, '.FactItemInformation'))
    except Exception: pass
    txt=safe_get_text(driver, By.CSS_SELECTOR, '.SecondaryFacts')
    if txt and any(k in txt for k in ['On Campus','Online','Blended']): return txt
    return None

def get_credits():
    try:
        for art in driver.find_elements(By.CSS_SELECTOR, 'article.FactItem'):
            ttl=safe_get_text(art.find_element(By.CSS_SELECTOR, '.FactItemTitle'))
            if ttl and ttl.lower().startswith('credits'):
                return safe_get_text(art.find_element(By.CSS_SELECTOR, '.FactItemInformation'))
    except Exception: pass
    return None

def get_campus_location():
    try:
        for art in driver.find_elements(By.CSS_SELECTOR, 'article.FactItem'):
            ttl=safe_get_text(art.find_element(By.CSS_SELECTOR, '.FactItemTitle'))
            if ttl and 'campus location' in ttl.lower():
                items=[safe_get_text(li) for li in safe_find_all(By.CSS_SELECTOR, '.CampusLocationList li', ctx=art)]
                items=[x for x in items if x]; return items or None
    except Exception: pass
    return None

def get_disciplines():
    out=[]
    try:
        art=driver.find_element(By.CSS_SELECTOR, 'article.FactItem.Disciplines, article.Disciplines')
        for a in safe_find_all(By.CSS_SELECTOR, 'a', ctx=art):
            t=safe_get_text(a)
            if t: out.append(t)
    except Exception: pass
    return out

def get_gpa_class():
    try:
        scopes = driver.find_elements(By.CSS_SELECTOR, '#OtherRequirements, article#OtherRequirements, section#EntryRequirements, .Requirements, .RequirementsWrapper') or [driver]
        for sc in scopes:
            for sp in safe_find_all(By.CSS_SELECTOR, '.Score span, .Score', ctx=sc):
                t=safe_get_text(sp)
                if t and re.search(r'Lower|Upper|First|Second|GPA|Class', t, re.I): return t
    except Exception: pass
    return None

def get_requirements_text():
    out=[]
    for sel in ['article#OtherRequirements .OtherRequirementsContent','section#EntryRequirements','.RequirementsConditions','.Requirements, .RequirementsWrapper']:
        for box in safe_find_all(By.CSS_SELECTOR, sel):
            txt=safe_get_text(box)
            if txt and txt not in out: out.append(txt)
    for li in safe_find_all(By.CSS_SELECTOR, 'article#OtherRequirements li, section#EntryRequirements li'):
        t=safe_get_text(li)
        if t and t not in out: out.append(t)
    return out

def get_tuition():
    res={"original_amount":None,"original_currency":None,"display_amount":None,"display_currency":None,"display_duration":None}
    try:
        amount = driver.find_element(By.CSS_SELECTOR, '.Amount .js-currencyAmount, .TuitionFee .js-currencyAmount, .TuitionValue .js-currencyAmount')
        res["original_amount"]=safe_attr(amount,'data-original-amount')
        res["original_currency"]=safe_attr(amount,'data-currency')
        res["display_amount"]=safe_get_text(amount)
        cur=None
        for sel in ['.Amount .CurrencyType[data-currency-text]','.Amount .CurrencyType']:
            els=driver.find_elements(By.CSS_SELECTOR, sel)
            if els: cur=safe_get_text(els[0]); break
        res["display_currency"]=cur if cur else None
        els=driver.find_elements(By.CSS_SELECTOR, '.Amount .CurrencyType ~ .CurrencyType')
        if els: res["display_duration"]=safe_get_text(els[0])
    except Exception:
        res["display_amount"]=safe_get_text(driver, By.CSS_SELECTOR, '.TuitionValue')
    return res

def get_cost_of_living():
    out={"amount_min_display":None,"amount_max_display":None,"base_currency":None,"amount_min_base":None,"amount_max_base":None,"display_currency":None}
    try:
        costs=driver.find_element(By.CSS_SELECTOR, '.Costs')
        spans=costs.find_elements(By.CSS_SELECTOR, '.Amount span[data-amount]')
        if spans:
            out["base_currency"]=safe_attr(spans[0],'data-currency')
            out["amount_min_base"]=safe_attr(spans[0],'data-amount')
            out["amount_min_display"]=safe_get_text(spans[0])
            if len(spans)>1:
                out["amount_max_base"]=safe_attr(spans[1],'data-amount')
                out["amount_max_display"]=safe_get_text(spans[1])
        cur=None
        els=costs.find_elements(By.CSS_SELECTOR, '.CurrencyType, [data-currency-text]')
        if els: cur=safe_get_text(els[0])
        out["display_currency"]=cur
    except Exception: pass
    return out

def record_from_url(url):
    rec={"source_url":url,"degree_tag":None,"university_name":None,"program_name":None,"review_score":None,"review_count":None,"durations":[], \
         "start_dates":[], "language":None,"language_requirements":[], "delivery_type":None,"credits":None,"campus_location":[], \
         "disciplines":[], "gpa_class":None,"requirements_text":[], "tuition":None,"cost_of_living":None,"error":None}
    try:
        go(url)  # (CF aware)
        try: wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, 'h1.StudyTitleWrapper, .StudyTitle')))
        except Exception: pass
        rec["degree_tag"]=get_degree_tag()
        uni,prog=get_university_and_program()
        rec["university_name"],rec["program_name"]=uni,prog
        rv,rc=get_review()
        rec["review_score"],rec["review_count"]=rv,rc
        rec["durations"]=get_durations()
        rec["start_dates"]=get_start_dates()
        rec["language"]=get_language()
        rec["language_requirements"]=get_language_requirements()
        rec["delivery_type"]=get_delivery_type()
        rec["credits"]=get_credits()
        rec["campus_location"]=get_campus_location() or []
        rec["disciplines"]=get_disciplines()
        rec["gpa_class"]=get_gpa_class()
        rec["requirements_text"]=get_requirements_text()
        rec["tuition"]=get_tuition()
        rec["cost_of_living"]=get_cost_of_living()
    except Exception as e:
        rec["error"]=f"{type(e).__name__}: {_norm_text(str(e))}"
    return rec

def _to_text(x):
    if x is None: return None
    if isinstance(x, list):
        parts=[]
        for v in x:
            if v is None: continue
            parts.append(json.dumps(v, ensure_ascii=False) if isinstance(v,(dict,list)) else _norm_text(v))
        return " | ".join([p for p in parts if p])
    if isinstance(x, dict):
        return json.dumps(x, ensure_ascii=False)
    return _norm_text(x)

def df_from_records(records):
    rows=[]
    for rec in records:
        rows.append({
            "source_url": rec.get("source_url"),
            "degree_tag": rec.get("degree_tag"),
            "university_name": rec.get("university_name"),
            "program_name": rec.get("program_name"),
            "review_score": rec.get("review_score"),
            "review_count": rec.get("review_count"),
            "durations": _to_text(rec.get("durations")),
            "start_dates": _to_text(rec.get("start_dates")),
            "language": rec.get("language"),
            "language_requirements": _to_text(rec.get("language_requirements")),
            "delivery_type": rec.get("delivery_type"),
            "credits": rec.get("credits"),
            "campus_location": _to_text(rec.get("campus_location")),
            "disciplines": _to_text(rec.get("disciplines")),
            "gpa_class": rec.get("gpa_class"),
            "requirements_text": _to_text(rec.get("requirements_text")),
            "tuition": json.dumps(rec.get("tuition"), ensure_ascii=False) if rec.get("tuition") is not None else None,
            "cost_of_living": json.dumps(rec.get("cost_of_living"), ensure_ascii=False) if rec.get("cost_of_living") is not None else None,
            "error": rec.get("error"),
        })
    return pd.DataFrame(rows)

def write_to_files(new_records, final_flush=False):
    if not new_records: return pd.DataFrame()
    Path(EXCEL_PATH).parent.mkdir(parents=True, exist_ok=True)
    df_new = df_from_records(new_records)
    if Path(EXCEL_PATH).exists():
        try: df_old = pd.read_excel(EXCEL_PATH, sheet_name="results", engine="openpyxl")
        except Exception: df_old = pd.DataFrame()
        df_all = pd.concat([df_old, df_new], ignore_index=True)
        if "source_url" in df_all.columns:
            df_all = df_all.drop_duplicates(subset=["source_url"], keep="first")
    else:
        df_all = df_new
    with pd.ExcelWriter(EXCEL_PATH, engine="openpyxl") as writer:
        df_all.to_excel(writer, index=False, sheet_name="results")
        writer.sheets["results"].freeze_panes = "A2"
    df_all.to_csv(CSV_PATH, index=False, encoding="utf-8-sig")
    print(("✔ Final" if final_flush else "↻ Ara"), f"Excel/CSV yazıldı. Toplam satır: {len(df_all)}")
    return df_all

all_results=[]; errors=[]; visited=set()
total_pages=0; last_saved_idx=0

go(URL_START)  # (CF aware)

while True:
    total_pages += 1
    try:
        wait.until(EC.presence_of_all_elements_located((By.CSS_SELECTOR, 'ul.SearchResultsList li.SearchResultItem')))
    except Exception: pass

    scroll_lazy_until_cards_stop()

    next_href=None
    try:
        # Altlara inip buton href'inin dolmasını garanti et
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);"); time.sleep(0.5)
        btns = driver.find_elements(By.CSS_SELECTOR, 'a.NextButton.NavigatorButton')
        if btns:
            nb = btns[0]
            cls = (safe_attr(nb,"class") or "")
            href = safe_attr(nb,"href")
            if (("Disabled" in cls) or (not href)):
                next_href=None
            else:
                next_href = urljoin(driver.current_url, href)
        info = safe_get_text(driver, By.CSS_SELECTOR, 'p.SeeMoreLabelVar1') or ""
        if info: print(f"[Sayfa {total_pages}] {info}")
    except Exception:
        next_href=None

    page_urls=[]
    for li in driver.find_elements(By.CSS_SELECTOR, 'ul.SearchResultsList > li.SearchResultItem'):
        if is_ad_placeholder(li): continue
        for a in li.find_elements(By.CSS_SELECTOR, 'a.SearchStudyCard.js-bestFitStudycard.js-studyCardExperiment.HoverEffect[href*="/studies/"]'):
            href = safe_attr(a,"href")
            if href and "/studies/" in href:
                href = href.split("#")[0]
                href = urljoin(driver.current_url, href)
                if href not in visited:
                    visited.add(href)
                    page_urls.append(href)

    page_results=[]
    for u in page_urls:
        rec = record_from_url(u)
        page_results.append(rec)
        if rec.get("error"): errors.append({"source_url":u,"error":rec.get("error")})

    all_results.extend(page_results)

    if total_pages % 5 == 0:
        chunk = all_results[last_saved_idx:]
        write_to_files(chunk, final_flush=False)
        last_saved_idx = len(all_results)

    if not next_href:
        break
    try:
        go(next_href)  # (CF aware)
        wait.until(EC.presence_of_all_elements_located((By.CSS_SELECTOR, 'ul.SearchResultsList li.SearchResultItem')))
    except Exception:
        break

if last_saved_idx < len(all_results):
    write_to_files(all_results[last_saved_idx:], final_flush=True)
else:
    if Path(EXCEL_PATH).exists():
        try:
            df_all = pd.read_excel(EXCEL_PATH, sheet_name="results", engine="openpyxl")
            df_all.to_csv(CSV_PATH, index=False, encoding="utf-8-sig")
        except Exception: pass

print("\n--- Özet ---")
print(f"Toplam sayfa: {total_pages}")
print(f"Toplam kayıt: {len(all_results)}")
print(f"Excel yolu  : {EXCEL_PATH}")
print(f"CSV yolu    : {CSV_PATH}")
print(f"Hata sayısı : {len(errors)}")
for i,e in enumerate(errors[:3],1):
    print(f"#{i} Hata: {e['source_url']} -> {e['error']}")
print("\nÖrnek ilk 3 kayıt:")
print(json.dumps(all_results[:3], ensure_ascii=False, indent=2))


[Sayfa 1] 18 of 45


KeyboardInterrupt: 